In [1]:
%load_ext autoreload
%autoreload 2
%reset -f

In [2]:
from pathlib import Path
import os
from os.path import join
import sys
import sqlite3
# Find KPIHub root (directory containing lib/tables), then import lib.* as a package.
_here = Path.cwd().resolve()
_root = next((p for p in [_here, *list(_here.parents)[:12]] if (p / "lib" / "tables").is_dir()), None)
if _root is None:
    raise FileNotFoundError(
        f"Could not find KPIHub root (folder containing lib/tables). cwd={Path.cwd()!r}"
    )
sys.path.insert(0, str(_root))
os.chdir(_root)

from locallib.picarrodb import *
from locallib.slack import *
from locallib.etl import Loggers
from locallib.pandas import *

from lib.tables.IngesterTables import *
from lib.handlers.CustomerHandler import *
from lib.config import *
from lib.KPIHubConnection import *
from lib.query.bank import *

from datetime import date
from datetime import timedelta

EU1_Conn created successfully
EU2_Conn created successfully
DataHub_Conn created successfully
US_Conn created successfully
EU1_PROD_Conn created successfully
EU2_PROD_Conn created successfully


In [3]:
conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()
KPI_META_KEYS = [
        'ReportAssetLengthKm',
        'AssetCoveredLengthKm',
        'DistributionPipeKm',
        'DistributionPipeCoveredKm',
        'CumulativeAssetCoveredLengthKm',
        'ServicePipeKm',
        'ServicePipeCoveredKm',
        'ReportCount',
        'DaysCount',
        'FOVMain',
        'SurveyDurationHours',
        'TargetDurationHours',
        'CustomerUtilization',
        'StarndardUtilization',
        'TotalSurveyors',
        'ProductivityPerSurveyor',
        'SurveyCount',
        'AvgSpeedKm',
        'SurveysCarDay',
        'IdleTime',
        'TotalDrivenLengthKm',
        'DrivingRatio',
        'NightDrivenLength',
        'DayDrivenLength',
        'NightRatio',
        'DayRatio',
        'PeakAboveSATCount',
        'LisaCount',
        'EmissionRate',
        'B0Count',
        'B1Count',
        'Bm1Count',
        'Bm2Count',
        'NGCount',
        'PGCount',
        'Not_NGCount',
        'LisaDensity',
        'InstatanoeusEmission',
        'B0Density',
        'B1Density',
        'Bm1Density',
        'Bm2Density',
        'B0Share',
        'B1Share',
        'Bm1Share',
        'Bm2Share',
        'NGShare',
        'PGShare',
        'Not_NGShare',
    ]


In [4]:
query = f"""DROP VIEW IF EXISTS Yearly_KPI;"""
cursor.execute(query)
conn.commit()


In [5]:
query = """
CREATE VIEW IF NOT EXISTS Yearly_KPI AS
SELECT
    kd.Year,
    kd.PeriodValue,
    kc.Name AS CustomerName,
    kd.BoundaryRegion,
    """ + ",\n    ".join([f"SUM(CASE WHEN kd.KPIId = '{kpi}' THEN kd.Value END) AS [{kpi}]" for kpi in KPI_META_KEYS]) + """
FROM KPI_Data kd
LEFT JOIN KPI_Customer kc ON kd.CustomerId = kc.CustomerId
WHERE kd.PeriodType = 'Year'
GROUP BY kd.Year, kd.PeriodValue, kd.CustomerId, kc.Name, kd.BoundaryRegion;"""
cursor.execute(query)
conn.commit()

In [6]:
print(query)


CREATE VIEW IF NOT EXISTS Yearly_KPI AS
SELECT
    kd.Year,
    kd.PeriodValue,
    kc.Name AS CustomerName,
    kd.BoundaryRegion,
    SUM(CASE WHEN kd.KPIId = 'ReportAssetLengthKm' THEN kd.Value END) AS [ReportAssetLengthKm],
    SUM(CASE WHEN kd.KPIId = 'AssetCoveredLengthKm' THEN kd.Value END) AS [AssetCoveredLengthKm],
    SUM(CASE WHEN kd.KPIId = 'DistributionPipeKm' THEN kd.Value END) AS [DistributionPipeKm],
    SUM(CASE WHEN kd.KPIId = 'DistributionPipeCoveredKm' THEN kd.Value END) AS [DistributionPipeCoveredKm],
    SUM(CASE WHEN kd.KPIId = 'CumulativeAssetCoveredLengthKm' THEN kd.Value END) AS [CumulativeAssetCoveredLengthKm],
    SUM(CASE WHEN kd.KPIId = 'ServicePipeKm' THEN kd.Value END) AS [ServicePipeKm],
    SUM(CASE WHEN kd.KPIId = 'ServicePipeCoveredKm' THEN kd.Value END) AS [ServicePipeCoveredKm],
    SUM(CASE WHEN kd.KPIId = 'ReportCount' THEN kd.Value END) AS [ReportCount],
    SUM(CASE WHEN kd.KPIId = 'DaysCount' THEN kd.Value END) AS [DaysCount],
    SUM(CASE WH

In [7]:
query = "SELECT * FROM Yearly_KPI WHERE Year = 2025;"
df = pd.read_sql_query(query, conn)
conn.close()
df

,Year,PeriodValue,CustomerName,BoundaryRegion,ReportAssetLengthKm,AssetCoveredLengthKm,DistributionPipeKm,DistributionPipeCoveredKm,CumulativeAssetCoveredLengthKm,ServicePipeKm,...,B1Density,Bm1Density,Bm2Density,B0Share,B1Share,Bm1Share,Bm2Share,NGShare,PGShare,Not_NGShare
0,2025,None,ITALGAS,None,62369.76,58400.60,62369.76,58400.60,None,0.0,...,0.01,0.82,0.20,15.03,0.53,67.58,16.86,54.13,27.12,18.75
1,2025,None,ITALGAS,Abruzzo,2164.05,2030.82,2164.05,2030.82,None,0.0,...,0.01,0.78,0.20,15.02,0.51,67.28,17.19,61.52,25.94,12.54
2,2025,None,ITALGAS,Calabria,5246.67,4832.78,5246.67,4832.78,None,0.0,...,0.00,0.46,0.08,19.63,0.61,68.07,11.69,54.82,30.02,15.16
3,2025,None,ITALGAS,Campania,4448.61,4152.67,4448.61,4152.67,None,0.0,...,0.01,1.37,0.39,12.03,0.46,68.22,19.29,55.88,28.80,15.32
4,2025,None,ITALGAS,Centro Italia,4567.22,4240.21,4567.22,4240.21,None,0.0,...,0.01,0.85,0.19,17.95,1.01,66.36,14.68,63.89,24.26,11.85
5,2025,None,ITALGAS,Lazio Sud,3354.68,3133.05,3354.68,3133.05,None,0.0,...,0.00,0.73,0.20,13.54,0.41,67.22,18.82,67.29,15.66,17.05
6,2025,None,ITALGAS,Liguria,3905.01,3623.95,3905.01,3623.95,None,0.0,...,0.00,0.66,0.16,14.14,0.29,69.15,16.42,55.36,29.43,15.21
7,2025,None,ITALGAS,Lombardia Nord-Est,1815.86,1697.45,1815.86,1697.45,None,0.0,...,0.01,0.89,0.22,9.09,0.43,72.44,18.04,35.92,27.11,36.97
8,2025,None,ITALGAS,Lombardia Nord-Ovest,974.40,905.84,974.40,905.84,None,0.0,...,0.00,0.82,0.22,12.37,0.28,68.93,18.42,45.31,25.18,29.51
9,2025,None,ITALGAS,Lombardia Sud,1322.58,1187.84,1322.58,1187.84,None,0.0,...,0.01,0.75,0.18,15.15,0.68,67.65,16.52,34.63,27.14,38.23


In [8]:
Query(query = f"SELECT * FROM KPI_Data WHERE CustomerId IN (SELECT CustomerId FROM KPI_Customer WHERE Name = 'ITALGAS')").execute(KPIHub_Conn)

,Id,KPIId,CustomerId,BoundaryRegion,Year,PeriodType,PeriodValue,Value,DataType,LastUpdated
0,FOVMain_ITALGAS_Y2025_W3,FOVMain,CFFB9000-94BD-BA72-B352-39EBA962116D,None,2025,Week,3.0,89.15,None,2026-07-21 20:29:15.547937
1,FOVMain_ITALGAS_Y2025_W4,FOVMain,CFFB9000-94BD-BA72-B352-39EBA962116D,None,2025,Week,4.0,92.95,None,2026-07-21 20:29:15.547937
2,FOVMain_ITALGAS_Y2025_W5,FOVMain,CFFB9000-94BD-BA72-B352-39EBA962116D,None,2025,Week,5.0,94.67,None,2026-07-21 20:29:15.547937
3,FOVMain_ITALGAS_Y2025_W6,FOVMain,CFFB9000-94BD-BA72-B352-39EBA962116D,None,2025,Week,6.0,92.5,None,2026-07-21 20:29:15.547937
4,FOVMain_ITALGAS_Y2025_W7,FOVMain,CFFB9000-94BD-BA72-B352-39EBA962116D,None,2025,Week,7.0,95.76,None,2026-07-21 20:29:15.547937
...,...,...,...,...,...,...,...,...,...,...
25723,Not_NGShare_ITALGAS_Y2026_BSiciliaEst,Not_NGShare,CFFB9000-94BD-BA72-B352-39EBA962116D,Sicilia Est,2026,Year,NaN,15.98,None,2026-07-21 20:29:20.825426
25724,Not_NGShare_ITALGAS_Y2026_BSiciliaOvest,Not_NGShare,CFFB9000-94BD-BA72-B352-39EBA962116D,Sicilia Ovest,2026,Year,NaN,37.59,None,2026-07-21 20:29:20.825426
25725,Not_NGShare_ITALGAS_Y2026_BTorino,Not_NGShare,CFFB9000-94BD-BA72-B352-39EBA962116D,Torino,2026,Year,NaN,20.7,None,2026-07-21 20:29:20.825426
25726,Not_NGShare_ITALGAS_Y2026_BVenetoNord-Friuli,Not_NGShare,CFFB9000-94BD-BA72-B352-39EBA962116D,Veneto Nord - Friuli,2026,Year,NaN,23.9,None,2026-07-21 20:29:20.825426
